# 05d. G 표본 추정 (판정기준서 v1.3, D67·D68, 2026-11-24)

- 상위 30개 밖 추천 상품에서 원천(데이터 등록) 등급별 층화 무작위 200개 (시드 20261124), 상위 30과 겹치면 제외.
- 조회: `src.latest_grade.lookup` (상품명만, 요청 간격 1초 이상, FunETF 참고). 캐시 `data/llm_cache/latest_grade_sample_lookup_20261124.csv`(git 제외)에 이어 쓴다.
- 추정: 층화 추정, 95% 신뢰구간(정규 근사, 유한 모집단 수정). 함수와 애매한 경우 처리는 `src/g_sample.py`.
- 숫자만 둔다. 해석은 쓰지 않는다.

데이터: AI Hub(한국지능정보사회진흥원) 사업 결과

In [ ]:
import sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
pd.set_option("display.width", 220)
from src.load_cot import load_cot, load_products
from src.judge import prepare
from src.g_sample import frame, draw, lookup_resumable, per_product, estimate, SEED, N_SAMPLE
AS_OF, CHECKED = "2026-11-24", "2026-11-24"
cot = load_cot(); prod = load_products()
ctx = prepare(cot, prod, AS_OF)
top = pd.read_csv(ROOT / "outputs" / "latest_grade_check.csv", encoding="utf-8-sig", keep_default_na=False)
g4top = pd.read_csv(ROOT / "outputs" / "latest_grade_rejudge.csv", encoding="utf-8-sig", keep_default_na=False)
fr, info = frame(ctx["recs"], set(top["상품명"]))
print(info); print("층 크기:", fr.groupby("층").size().to_dict())
sample = draw(fr, N_SAMPLE, SEED)
print("표본:", len(sample), "/ 층별:", sample.groupby("층").size().to_dict())

## 1. 조회 (상품명만, 1초 이상 간격)

In [ ]:
cache = ROOT / "data" / "llm_cache" / "latest_grade_sample_lookup_20261124.csv"
looked = lookup_resumable(sample["상품명"].tolist(), cache, CHECKED, delay=1.0)
s = per_product(sample, looked, ctx["recs"], ctx["cots"], ctx["range"])
print("조회 상태:", s["상태"].value_counts().to_dict())
print("층별 조회 가능·등급 변경·부적합 추천:")
print(s.groupby("층").agg(표본=("상품명", "size"), 모집단=("층_N", "first"), 조회가능=("조회 가능", "sum"),
                         등급변경=("등급 변경", "sum"), 부적합추천=("최신 기준 부적합 추천", "sum")).to_string())

## 2. 전체 추정과 95% 신뢰구간

In [ ]:
top_g4 = int((g4top["G4 태그"] != "").sum())
est = estimate(s, top_g4)
print(est.to_string(index=False))
cols = ["상품명", "층", "층_N", "층_n", "등급", "최신 등급", "상태", "조회 가능", "등급 변경", "최신 기준 부적합 추천",
        "데이터 범위 밖 추천", "조회 이름", "조회 클래스", "출처 순위", "URL", "확인일"]
s.rename(columns={"등급": "데이터 등급"}).reindex(columns=[c if c != "등급" else "데이터 등급" for c in cols])  .to_csv(ROOT / "outputs" / "latest_grade_sample.csv", index=False, encoding="utf-8-sig")
est.to_csv(ROOT / "outputs" / "latest_grade_sample_estimate.csv", index=False, encoding="utf-8-sig")

## 2b. 운용사 공시 확인 (D68)
표본 추정을 보고에 쓰는 기준(미리 고정): 등급이 바뀐 표본 상품 중 추천 수 상위 5개를 운용사 공시로 확인해 80% 이상 일치.
사람 확인 결과는 `data/review/등급변경_확인.csv`(비고에 'G 표본')에서 읽는다.

In [ ]:
rv = pd.read_csv(ROOT / "data" / "review" / "등급변경_확인.csv", encoding="utf-8-sig", keep_default_na=False)
rv = rv[rv["비고"].str.contains("G 표본")]
chk = rv.merge(s[["상품명", "최신 등급"]].rename(columns={"최신 등급": "조회 등급"}), on="상품명", how="left")
chk["일치"] = (chk["사람 확인"] == "최신 등급 맞음") & (pd.to_numeric(chk["조회 등급"]) == pd.to_numeric(chk["최신 등급"]))
n_chk, n_ok = len(chk), int(chk["일치"].sum())
print(chk[["상품명", "데이터 등급", "최신 등급", "조회 등급", "일치"]].to_string(index=False))
print(f"일치 {n_ok}/{n_chk}, 기준 80% 이상:", n_chk >= 5 and n_ok / n_chk >= 0.8)

## 3. scorecard 저장

In [ ]:
from src.report import update_scorecard
rows = [{"지표": f"표본 설계: {k}", "전체": v, "단위": "상품", "출처": "05d_G표본추정", "근거 결정": "D67", "기준일": AS_OF} for k, v in info.items()]
rows.append({"지표": "표본 상품 수 (시드 20261124)", "전체": len(s), "단위": "상품", "출처": "05d_G표본추정", "근거 결정": "D67", "기준일": AS_OF})
for _, r in est.iterrows():
    rows.append({"지표": f"{r['지표']} (95% CI {r['95% 하한']}–{r['95% 상한']})", "전체": r["추정치"], "단위": r["단위"],
                 "출처": "05d_G표본추정", "근거 결정": f"D67, {r['비고']}", "기준일": AS_OF})
rows.append({"지표": "운용사 공시 확인: 일치 상품 (확인 상품 중, 기준 80% 이상)", "전체": f"{n_ok}/{n_chk}", "단위": "상품",
             "출처": "05d_G표본추정", "근거 결정": "D68", "기준일": AS_OF})
rows.append({"지표": "보고 문구: 최신 기준 부적합 추천 최소 약 1,200건 (표본 추정, 95% CI 545–1,847)",
             "전체": round(float(est.iloc[-1]["추정치"]), 1), "단위": "추천", "출처": "05d_G표본추정", "근거 결정": "D68", "기준일": AS_OF})
update_scorecard(rows, "G 표본 추정 (05d, D67·D68)")
print("scorecard 저장:", len(rows), "행")

---
데이터: AI Hub(한국지능정보사회진흥원) 사업 결과